In [46]:
import pandas as pd

# Use this one for all three links
# The other scripts are alternative explorations.

# --- CONFIG ---
INPUT_FILE = "nodes_and_links.ods"   # change path if needed
FOCUS = "keywords"
SHEET_NAME = "{}_links".format(FOCUS)
DELIM = ";"            # your requested delimiter
OUTPUT_FILE_1 = "{}_expanded_1.ods".format(FOCUS)  #


# Load the sheet
df = pd.read_excel("nodes_and_links.ods", sheet_name="{}_links".format(FOCUS), engine="odf")

# assume ID is column 0, the two multi-line columns are column 1 and 2
id_col = df.columns[0]
col2 = df.columns[1]
#col3 = df.columns[2]

# --- prepare split lists: handle NaN, split, and strip whitespace ---
def safe_split(series, delim):
    return (
        series.fillna("")                        # replace NaN with empty string
              .astype(str)                       # in case there are numbers
              .apply(lambda s: [p.strip() for p in s.split(delim)])  # split & strip
    )

df[col2 + "_split"] = safe_split(df[col2], DELIM)
#df[col3 + "_split"] = safe_split(df[col3], DELIM)

# --- function to expand a single row into a DataFrame ---
def expand_row(row):
    a = row[col2 + "_split"]
    #b = row[col3 + "_split"]
    # ensure lists
    if not isinstance(a, list): a = [a]
    #if not isinstance(b, list): b = [b]
    max_len = len(a)
    # pad shorter with empty strings so they align
    a = a + [""] * (max_len - len(a))
    #b = b + [""] * (max_len - len(b))
    return pd.DataFrame({
        id_col: [row[id_col]] * max_len,
        col2: a,
        #col3: b
    })

# --- APPLY row-by-row and collect into a LIST (important: list, not Series) ---
expanded_dfs = [expand_row(row) for _, row in df.iterrows()]   # list comprehension

# If you prefer apply(), ensure you convert to list:
# expanded_dfs = df.apply(expand_row, axis=1).tolist()

# --- CONCAT the list of DataFrames ---
expanded_df = pd.concat(expanded_dfs, ignore_index=True)

# --- OPTIONAL: show rows where original split lengths differed (warning) ---
#mismatch_mask = df[col2 + "_split"].apply(len) != df[col3 + "_split"].apply(len)
#if mismatch_mask.any():
 #   print("Warning: some rows had differing number of items in Col2")
 #  print("Rows with mismatches (index, len_col2:")
 #   for i in df[mismatch_mask].index:
 #       print(i, len(df.at[i, col2 + "_split"]))

# --- SAVE result ---
expanded_df.to_excel(OUTPUT_FILE_1, index=False)
print("Expanded sheet written to:", OUTPUT_FILE_1)


Expanded sheet written to: keywords_expanded_1.ods


In [37]:
# For addressing topics

import pandas as pd
OUTPUT_FILE_2 = "{}_expanded_2.ods".format(FOCUS)

# Load your spreadsheet
df = pd.read_excel(OUTPUT_FILE_1)

# Split the column
df[['{}_a'.format(FOCUS), '{}_b'.format(FOCUS)]] = df["{}".format(FOCUS)].str.split(':', expand=True)

# Save it back
df.to_excel(OUTPUT_FILE_2, index=False)
print("Expanded sheet written to:", OUTPUT_FILE_2)

Expanded sheet written to: topics_expanded_2.ods


In [39]:
# For addressing topics

import pandas as pd

# --- CONFIG ---
INPUT_FILE = "nodes_and_links.ods"
SHEET_NAME = "Sheet1"
DELIM = ","
OUTPUT_FILE_3 = f"{FOCUS}_expanded_3.ods"

# --- LOAD DATA ---
df = pd.read_excel(OUTPUT_FILE_2, sheet_name=SHEET_NAME, engine="odf")
df.columns = df.columns.str.strip()

id_col = df.columns[0]
col2 = df.columns[2]   # e.g. your "middle" column
col3 = df.columns[3]   # e.g. the column you want to explode

# --- SPLIT the explode column safely ---
def safe_split(series, delim):
    return (
        series.fillna("")
              .astype(str)
              .apply(lambda s: [p.strip() for p in s.split(delim) if p.strip() != ""])
    )

df[f"{col3}_split"] = safe_split(df[col3], DELIM)

# --- EXPLODE just col3, keeping others aligned ---
df_exploded = df.explode(f"{col3}_split", ignore_index=True)

# --- Rename col3 to hold the exploded values (cleaner) ---
df_exploded[col3] = df_exploded[f"{col3}_split"]
df_exploded = df_exploded.drop(columns=[f"{col3}_split"])

# --- SAVE RESULT ---
df_exploded.to_excel(OUTPUT_FILE_3, index=False, engine="odf")
print("Expanded sheet written to:", OUTPUT_FILE_3)



Expanded sheet written to: topics_expanded_3.ods


In [27]:
import os
import glob

# Define a pattern for unwanted files
temp_patterns = ["*_1.ods", "*_2.ods"]

for pattern in temp_patterns:
    for f in glob.glob(pattern):
        try:
            os.remove(f)
            print(f"Deleted: {f}")
        except Exception as e:
            print(f"Could not delete {f}: {e}")


Deleted: topics_expanded_1.ods
Deleted: topics_expanded_2.ods


In [47]:
import pandas as pd

# specifically for the second pass on key words with the ',' delimeter

# --- CONFIG ---
INPUT_FILE = "keywords_expanded_1.ods"   # change path if needed
FOCUS = "keywords"
SHEET_NAME = "Sheet1"
DELIM = ","            # your requested delimiter
OUTPUT_FILE_4 = "{}_expanded_4.ods".format(FOCUS)  #


# Load the sheet
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1", engine="odf")

# assume ID is column 0, the two multi-line columns are column 1 and 2
id_col = df.columns[0]
col2 = df.columns[1]
#col3 = df.columns[2]

# --- prepare split lists: handle NaN, split, and strip whitespace ---
def safe_split(series, delim):
    return (
        series.fillna("")                        # replace NaN with empty string
              .astype(str)                       # in case there are numbers
              .apply(lambda s: [p.strip() for p in s.split(delim)])  # split & strip
    )

df[col2 + "_split"] = safe_split(df[col2], DELIM)
#df[col3 + "_split"] = safe_split(df[col3], DELIM)

# --- function to expand a single row into a DataFrame ---
def expand_row(row):
    a = row[col2 + "_split"]
    #b = row[col3 + "_split"]
    # ensure lists
    if not isinstance(a, list): a = [a]
    #if not isinstance(b, list): b = [b]
    max_len = len(a)
    # pad shorter with empty strings so they align
    a = a + [""] * (max_len - len(a))
    #b = b + [""] * (max_len - len(b))
    return pd.DataFrame({
        id_col: [row[id_col]] * max_len,
        col2: a,
        #col3: b
    })

# --- APPLY row-by-row and collect into a LIST (important: list, not Series) ---
expanded_dfs = [expand_row(row) for _, row in df.iterrows()]   # list comprehension

# If you prefer apply(), ensure you convert to list:
# expanded_dfs = df.apply(expand_row, axis=1).tolist()

# --- CONCAT the list of DataFrames ---
expanded_df = pd.concat(expanded_dfs, ignore_index=True)

# --- OPTIONAL: show rows where original split lengths differed (warning) ---
#mismatch_mask = df[col2 + "_split"].apply(len) != df[col3 + "_split"].apply(len)
#if mismatch_mask.any():
 #   print("Warning: some rows had differing number of items in Col2")
 #  print("Rows with mismatches (index, len_col2:")
 #   for i in df[mismatch_mask].index:
 #       print(i, len(df.at[i, col2 + "_split"]))

# --- SAVE result ---
expanded_df.to_excel(OUTPUT_FILE_4, index=False)
print("Expanded sheet written to:", OUTPUT_FILE_4)

Expanded sheet written to: keywords_expanded_4.ods
